# Segmentation des plans d'eau (Sentinel-2) — entraînement sur Kaggle (GPU)

1. Dans ce notebook Kaggle, clique **Add Input** et ajoute le dataset `franciscoescobar/satellite-images-of-water-bodies`.
   **Important** : si tu as importé ce fichier `.ipynb` (File > Import Notebook), le dataset n'est
   **pas** attaché automatiquement — il faut refaire Add Input, sinon la 2e cellule s'arrête avec un message explicite.
2. Dans le panneau de droite, mets **Accelerator = GPU T4 x2** (ou P100).
3. Utilise **Save Version > Save & Run All (Commit)** pour un Output fiable (~15-20 min).
4. Télécharge depuis l'Output :
   - `model.pt` → `models/model.pt` du projet local (contient aussi le seuil optimisé et les métriques)
   - `train_metrics.json` → `reports/train_metrics.json` (scores val/test à citer dans le rapport)
   - (optionnel) `mlflow.db` et `mlruns/` → à la racine du projet, puis
     `mlflow ui --backend-store-uri sqlite:///mlflow.db` pour voir les courbes
5. En local : `dvc repro` pour évaluer le nouveau modèle sur les vrais barrages tunisiens (`data/eval/`).

Changements par rapport à la v1 : split reproductible train/val/test (graine fixe), augmentations
(flips, rotations 90°, bandes no-data noires étiquetées « terre »), perte Dice + BCE, 30 epochs avec
cosine LR, sauvegarde du **meilleur** epoch (et non du dernier), IoU calculé sur tout le jeu (et non en
moyenne par batch), seuil de décision optimisé sur la validation puis score final sur un test jamais vu.

In [ ]:
!pip install -q segmentation-models-pytorch mlflow

In [ ]:
import json
import os
import random

import numpy as np
import segmentation_models_pytorch as smp
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_ROOT = '/kaggle/input'


def print_tree(root, max_depth=4):
    for dirpath, dirnames, files in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth >= max_depth:
            dirnames[:] = []
        print('  ' * depth + (os.path.basename(dirpath) or dirpath) + '/', f'({len(files)} fichiers)' if files else '')


def find_dir(root, name_options):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() in name_options:
                return os.path.join(dirpath, d)
    return None


if not os.path.isdir(INPUT_ROOT) or not os.listdir(INPUT_ROOT):
    raise FileNotFoundError(
        'Aucun dataset attaché (' + INPUT_ROOT + ' est vide). Dans le panneau de droite : '
        '"Add Input" > cherche "satellite-images-of-water-bodies" (franciscoescobar) > Add, '
        'puis relance toutes les cellules.'
    )

DATA_ROOT = find_dir(INPUT_ROOT, {'images', 'image'})
MASK_ROOT = find_dir(INPUT_ROOT, {'masks', 'mask'})
if DATA_ROOT is None or MASK_ROOT is None:
    print('Contenu de', INPUT_ROOT, ':')
    print_tree(INPUT_ROOT)
    raise FileNotFoundError(
        'Dossiers Images/Masks introuvables sous ' + INPUT_ROOT + ' (arborescence ci-dessus). '
        'Vérifie que le dataset attaché est bien "satellite-images-of-water-bodies", '
        'ou fixe DATA_ROOT / MASK_ROOT à la main.'
    )

# Pair images and masks by name without extension, so a .jpg image still
# finds its .png mask, and any image without a mask is skipped (and counted).
masks_by_stem = {os.path.splitext(f)[0]: f for f in os.listdir(MASK_ROOT)}
PAIRS = sorted(
    (f, masks_by_stem[os.path.splitext(f)[0]])
    for f in os.listdir(DATA_ROOT)
    if os.path.splitext(f)[0] in masks_by_stem
)
n_images = len(os.listdir(DATA_ROOT))
print('Images:', DATA_ROOT, '->', n_images, 'fichiers')
print('Masks:', MASK_ROOT, '->', len(masks_by_stem), 'fichiers')
print('Paires image/masque:', len(PAIRS), '| images sans masque ignorées:', n_images - len(PAIRS))
if not PAIRS:
    raise FileNotFoundError('Aucune image n\'a de masque du même nom dans ' + MASK_ROOT)

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 16

# Same plain resize as the API (src/serve.py) — no letterbox: the model
# must see at serving time exactly what it saw in training.
resize_image = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
resize_mask = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


class WaterBodiesDataset(Dataset):
    """With augment=True: 30% of the time a black no-data strip labelled
    land (v1 learned "dark = water" and flagged tile-edge no-data as water),
    then random flips and 90-degree rotations (orientation is arbitrary in
    satellite imagery, so these are free extra training data)."""

    def __init__(self, pairs, augment):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_name, mask_name = self.pairs[idx]
        image = transforms.functional.to_tensor(resize_image(Image.open(os.path.join(DATA_ROOT, image_name)).convert('RGB')))
        mask = (transforms.functional.to_tensor(resize_mask(Image.open(os.path.join(MASK_ROOT, mask_name)).convert('L'))) > 0.5).float()

        if self.augment:
            if random.random() < 0.3:
                width = random.randint(IMAGE_SIZE // 10, IMAGE_SIZE // 3)
                image[:, -width:, :] = 0
                mask[:, -width:, :] = 0
            if random.random() < 0.5:
                image, mask = image.flip(-1), mask.flip(-1)
            if random.random() < 0.5:
                image, mask = image.flip(-2), mask.flip(-2)
            k = random.randint(0, 3)
            image, mask = torch.rot90(image, k, (-2, -1)), torch.rot90(mask, k, (-2, -1))

        return normalize(image), mask


# Fixed-seed split, so the test set is the same on every run and never
# used for any decision (model selection uses val only).
perm = torch.randperm(len(PAIRS), generator=torch.Generator().manual_seed(SEED)).tolist()
n_holdout = int(0.15 * len(PAIRS))
test_files = [PAIRS[i] for i in perm[:n_holdout]]
val_files = [PAIRS[i] for i in perm[n_holdout:2 * n_holdout]]
train_files = [PAIRS[i] for i in perm[2 * n_holdout:]]

train_loader = DataLoader(WaterBodiesDataset(train_files, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(WaterBodiesDataset(val_files, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(WaterBodiesDataset(test_files, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('train:', len(train_files), 'val:', len(val_files), 'test:', len(test_files))

In [ ]:
import mlflow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

EPOCHS = 30
LR = 3e-4
# Plain Python floats: a NumPy scalar in the checkpoint makes torch.load
# (weights_only=True by default since PyTorch 2.6) refuse the file.
THRESHOLDS = [round(float(t), 2) for t in np.arange(0.3, 0.81, 0.05)]

# Whole network fine-tuned (pretrained ImageNet encoder is NOT frozen).
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=1).to(device)
dice = smp.losses.DiceLoss(mode='binary')
bce = smp.losses.SoftBCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def counts(loader, thresholds):
    """Dataset-level (tp, fp, fn) per threshold — an IoU from these is not
    skewed by batches with almost no water, unlike a mean of batch IoUs."""
    model.eval()
    result = {t: [0, 0, 0] for t in thresholds}
    with torch.no_grad():
        for images, masks in loader:
            probs = torch.sigmoid(model(images.to(device)))
            masks = masks.to(device).bool()
            for t in thresholds:
                pred = probs > t
                result[t][0] += (pred & masks).sum().item()
                result[t][1] += (pred & ~masks).sum().item()
                result[t][2] += (~pred & masks).sum().item()
    return result


def iou(tp, fp, fn):
    return tp / max(tp + fp + fn, 1)


# Recent MLflow refuses the old ./mlruns file backend: track in SQLite
# (artifacts still land in /kaggle/working/mlruns).
mlflow.set_tracking_uri('sqlite:////kaggle/working/mlflow.db')
mlflow.set_experiment('water-segmentation')
mlflow.start_run()
mlflow.log_params({
    'encoder': 'resnet34', 'encoder_weights': 'imagenet', 'encoder_frozen': False,
    'image_size': IMAGE_SIZE, 'epochs': EPOCHS, 'lr': LR, 'scheduler': 'cosine',
    'batch_size': BATCH_SIZE, 'loss': 'dice+bce', 'seed': SEED,
    'augmentation': 'nodata_strip,hflip,vflip,rot90',
    'n_train': len(train_files), 'n_val': len(val_files), 'n_test': len(test_files),
})

In [ ]:
BEST_WEIGHTS = '/kaggle/working/best_weights.pt'
best_iou, best_epoch = -1.0, -1

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = dice(outputs, masks) + bce(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_files)
    scheduler.step()

    val_iou = iou(*counts(val_loader, [0.5])[0.5])
    mlflow.log_metrics({'train_loss': train_loss, 'val_iou': val_iou}, step=epoch)
    if val_iou > best_iou:
        best_iou, best_epoch = val_iou, epoch + 1
        torch.save(model.state_dict(), BEST_WEIGHTS)
    print(f'epoch {epoch + 1}/{EPOCHS}  train_loss={train_loss:.4f}  val_iou={val_iou:.4f}  best={best_iou:.4f}@{best_epoch}')

model.load_state_dict(torch.load(BEST_WEIGHTS))
print('restored best epoch', best_epoch)

In [ ]:
# Threshold chosen on val, score reported on test (never seen before).
val_counts = counts(val_loader, THRESHOLDS)
threshold = max(THRESHOLDS, key=lambda t: iou(*val_counts[t]))
tp, fp, fn = counts(test_loader, [threshold])[threshold]

results = {
    'best_epoch': best_epoch,
    'threshold': threshold,
    'val_iou': round(iou(*val_counts[threshold]), 4),
    'test_iou': round(iou(tp, fp, fn), 4),
    'test_precision': round(tp / max(tp + fp, 1), 4),
    'test_recall': round(tp / max(tp + fn, 1), 4),
    'n_train': len(train_files), 'n_val': len(val_files), 'n_test': len(test_files),
    'seed': SEED,
}
print(json.dumps(results, indent=2))

METRICS_PATH = '/kaggle/working/train_metrics.json'
with open(METRICS_PATH, 'w') as f:
    json.dump(results, f, indent=2)

MODEL_PATH = '/kaggle/working/model.pt'
torch.save({
    'state_dict': model.state_dict(),
    'encoder_name': 'resnet34',
    'image_size': IMAGE_SIZE,
    'threshold': threshold,
    'metrics': results,
}, MODEL_PATH)

mlflow.log_params({'threshold': threshold, 'best_epoch': best_epoch})
mlflow.log_metrics({k: v for k, v in results.items() if k.startswith(('val_', 'test_'))})
mlflow.log_artifact(MODEL_PATH)
mlflow.log_artifact(METRICS_PATH)
mlflow.end_run()
print('saved', MODEL_PATH, 'and', METRICS_PATH)

In [ ]:
import matplotlib.pyplot as plt

model.eval()
images, masks = next(iter(test_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(images.to(device))).cpu()

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
display_images = (images * std + mean).clamp(0, 1)

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i in range(3):
    axes[i, 0].imshow(display_images[i].permute(1, 2, 0))
    axes[i, 0].set_title('image (test)')
    axes[i, 1].imshow(masks[i, 0], cmap='gray')
    axes[i, 1].set_title('masque réel')
    axes[i, 2].imshow(preds[i, 0] > threshold, cmap='gray')
    axes[i, 2].set_title(f'prédiction (seuil {threshold})')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_predictions.png')
plt.show()